---
## 🔍 Section 3: Attention & Context Window Behavior

### The "Lost in the Middle" Problem

Transformer attention is **not uniform**. Research consistently shows:

- Information at the **beginning** of a long context → retrieved reliably  
- Information at the **end** → retrieved reliably  
- Information **in the middle** of a long context → systematically underweighted

This is not a prompt quality issue. It's an architectural property of how attention matrices are computed.

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# DEMO: Lost-in-the-middle experiment
#
# We embed a "needle" fact in a long context at three positions:
#   - Beginning, Middle, End
# Then ask the model to retrieve it.
#
# KEY INSIGHT: If the model answers correctly at beginning/end
# but incorrectly in the middle — this is attention degradation,
# NOT a prompt problem.
# ─────────────────────────────────────────────────────────────

# A single specific fact we want the model to retrieve
needle_fact = "The SLA uptime guarantee is exactly 99.73%."

# Filler content — realistic-looking but generic contract language
# In a real system this would be actual retrieved documents
filler_paragraphs = [
    "Section 1: The Vendor shall maintain all production systems in accordance with industry best practices and applicable regulations.",
    "Section 2: Payment terms are net-30 from the date of invoice. Late payments accrue interest at 1.5% per month.",
    "Section 3: Either party may terminate this agreement with 90 days written notice for any reason.",
    "Section 4: All intellectual property developed under this agreement remains the property of the Client.",
    "Section 5: Disputes shall be resolved through binding arbitration in the jurisdiction of the Client's primary place of business.",
    "Section 6: The Vendor shall maintain comprehensive liability insurance of no less than $5 million per occurrence.",
    "Section 7: All data processed under this agreement shall comply with applicable data protection regulations.",
    "Section 8: The Vendor shall provide quarterly business reviews to assess performance metrics.",
]

def build_context(position: str) -> str:
    """
    Inserts the needle fact at beginning, middle, or end of filler content.
    Returns the full assembled context string.
    """
    mid = len(filler_paragraphs) // 2
    if position == "beginning":
        sections = [needle_fact] + filler_paragraphs
    elif position == "middle":
        sections = filler_paragraphs[:mid] + [needle_fact] + filler_paragraphs[mid:]
    else:  # end
        sections = filler_paragraphs + [needle_fact]
    return "\n\n".join(sections)


def ask_model(context: str, question: str) -> str:
    """
    Sends context + question to the model and returns its answer.
    Temperature=0 for deterministic output (no sampling noise).
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # Fast and cheap for experiments
        temperature=0,        # Fully deterministic — removes sampling as a variable
        messages=[
            {"role": "system", "content": "Answer the question using ONLY the provided document. Be concise."},
            {"role": "user", "content": f"Document:\n{context}\n\nQuestion: {question}"}
        ]
    )
    return response.choices[0].message.content


question = "What is the exact SLA uptime percentage mentioned in the document?"

print("Running Lost-in-the-Middle Experiment...\n")
print(f"Target fact: '{needle_fact}'")
print(f"Question: {question}")
print("=" * 60)

for position in ["beginning", "middle", "end"]:
    ctx = build_context(position)
    token_count = len(tiktoken.get_encoding("o200k_base").encode(ctx))
    answer = ask_model(ctx, question)
    print(f"\n📍 Needle position: {position.upper()} (~{token_count} tokens)")
    print(f"   Model answer: {answer}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# TODO 3 — Attention Probing (5 min)
#
# TASK:
#   Extend the experiment above with TWO similar facts
#   to trigger the "conflation" failure mode:
#   The model may confuse them when both appear in context.
#
#   Example: Two SLA clauses for different systems:
#     "The primary API SLA uptime guarantee is exactly 99.73%."
#     "The backup storage SLA uptime guarantee is exactly 95.0%."
#
#   Ask: "What is the SLA for the primary API?"
#   Ask: "What is the SLA for backup storage?"
#
#   Observe: Does the model correctly distinguish them?
#   What happens when you swap their positions?
# ─────────────────────────────────────────────────────────────

# --- YOUR CODE HERE ---
# needle_1 = "The primary API SLA uptime guarantee is exactly 99.73%."
# needle_2 = "The backup storage SLA uptime guarantee is exactly 95.0%."
# ...
pass